
# Persona-based Monthly Sales Forecast (LLM, Single-Turn, Soft Ratios)

이 노트북은 **페르소나 2,000명 JSON**과 **제품 리스트**를 바탕으로, 제품별로 **싱글턴 프롬프트**를 보내
월별 판매량을 추정하고, **사이즈/맛 비율을 자연스럽게(Soft) 근사**하도록 설계된 파이프라인입니다.

- 출력: `months_since_launch_1..12` 형식의 제출용 CSV
- 특징:
  - **설문조사 전문가 톤**의 단일 턴 프롬프트 (한국어, STRICT JSON only)
  - LLM이 **pack_size_preference / flavor_preference**(선택)를 함께 추정 → SKU 분배를 **부드럽게(soft)** 반영
  - **매크로 추세**, **이벤트/시즌성**, **라인(서브브랜드) 페널티** 반영
  - `OPENAI_API_KEY`가 없거나 SDK가 없으면 **시뮬레이션 모드**로 실행

> ⚠️ 실행 전, 아래 **Config** 셀에서 파일 경로를 당신의 환경에 맞게 바꿔주세요.


In [ ]:

from pathlib import Path

CFG = {
    "personas_json": "/mnt/data/personas_2000.json",         # <-- 수정: 페르소나 2000명 JSON
    "products_csv": "/mnt/data/data/product_info.csv",       # <-- 수정
    "sample_submission_csv": "/mnt/data/data/sample_submission.csv",  # <-- 수정
    "output_csv": "/mnt/data/submission_llm_persona.csv",
    "cache_dir": "/mnt/data/cache_llm",
    "use_llm": True,
    "simulate_if_no_key": True,
    "openai_model": "gpt-4o-mini",
    "requests_per_minute": 120,
    "temperature": 0.2,
    "top_p": 0.95,
    "persona_limit": None,   # 예: 200
    "product_limit": None,   # 예: 5
    # 12개월 매크로 경로 (2024-07 ~ 2025-06)
    "macro_path": [1.00, 1.0045, 1.0091, 1.0136, 1.0182, 1.0227, 1.0273, 1.0318, 1.0364, 1.0409, 1.0455, 1.05],
    # 이벤트(키워드, 월, 계수)
    "event_rules": [
        {"keyword": "참치", "month": "2024-09", "factor": 1.20},
        {"keyword": "맛참", "month": "2024-09", "factor": 1.12},
        {"keyword": "요거트", "month": "2025-01", "factor": 1.05},
        {"keyword": "참치", "month": "2025-02", "factor": 1.15},
        {"keyword": "맛참", "month": "2025-02", "factor": 1.08},
    ],
    # 서브라인 페널티 (제품명에 해당 문자열 포함 시 곱함)
    "line_penalties": {
        "동원맛참": 0.78
    },
    # Soft split 설정: 페르소나에서 나온 선호분포를 중심으로 SKU를 분배하되, 약한 priors(예: 8:2)를 살짝 섞음
    "soft_split": {
        "alpha_persona": 0.35,  # 0..1 (↑면 선호분포를 더 강하게 반영)
        "alpha_prior": 0.12,    # 0..1 (사전비율의 영향력)
        "priors": {
            "참치액_진": {"size": {"500g": 0.8, "900g": 0.2}},
            "참치액_순": {"size": {"500g": 0.8, "900g": 0.2}},
            "참치액_프리미엄": {"size": {"500g": 0.8, "900g": 0.2}},
            "동원맛참": {
                "size": {"90g": 0.55, "135g": 0.45},
                "flavor": {"고소": 0.6, "매콤": 0.4}
            },
        }
    },
}

# (선택) 환경변수 키 설정 예시
import os
# os.environ["OPENAI_API_KEY"] = "sk-..."  # 실제 키가 있을 때만 사용


In [ ]:

PROMPT_TEMPLATE = r'''너는 한국 소비재 분야의 **설문조사/표본조사 전문가**다. 출력은 반드시 **STRICT JSON only**로 한다.

[Panel 상황]
- 우리는 한국 소비자 2,000명(persona 패널)의 관찰을 기반으로, 아래에 제시한 1명의 페르소나를 대표 사례로 삼아 **인터뷰형 내적 추론**을 수행한다.
- 단일 턴(Single-turn)으로 판단하며, 외부 도구 사용 금지.
- 네 내부 인터뷰 문답은 출력하지 말고, 최종 결과(JSON)만 출력한다.

[가중치가 포함된 페르소나 속성]
{persona_block}

[제품 정보]
{product_block}

[과업]
아래의 12개 월(2024-07 ~ 2025-06)에 대해, 이 페르소나가 위 제품을 **월 기준**으로 구매할 **기대 구매수량(개)**을 추정하라.
추정 시 다음을 고려하라:
- 가중치가 큰 속성(연령/성별/소득/가구원수/구매채널/브랜드충성/할인민감/취식빈도/풍미선호/건강지향/라이프스타일 등)
- 가구원수와 포장단위(예: 500g vs 900g)의 상관
- 가격 민감도와 용량 선택의 트레이드오프
- 한국 명절/시즌성(예: 추석 2024-09, 설 2025-02)
- (중요) **싱글턴**: 이번 한 번의 입력만으로 최종 결론을 내려라.

[출력 스키마(숫자는 float)]
{{
  "buy_probability": 0.0_to_1.0,                 // 이 페르소나의 월간 '최소 1회 구매' 확률
  "expected_qty_per_purchase": 0.0_to_3.0,       // 구매 시 1회 기대 구매수량(개)
  "monthly_pattern": [                            // 길이 12, 각 월의 기대값(상대 척도 아님), 평균이 대략 buy_probability*expected_qty_per_purchase와 유사
    m1, m2, m3, m4, m5, m6, m7, m8, m9, m10, m11, m12
  ],
  "pack_size_preference": {{"EXAMPLE": 1.0}},   // (선택) 포장 용량 선호 분포(합=1)
  "flavor_preference": {{"EXAMPLE": 1.0}}       // (선택) 맛/라인업 선호 분포(합=1)
}}

[주의]
- 반드시 **유효한 JSON**만 출력하라(설명/문장/마크다운 금지).
- 값들은 음수가 될 수 없다.
- 소수점 허용.
- product_block 내부 "group_variants"에 이 라인의 사용 가능한 sizes/flavors가 제공될 수 있다. 제공된 값으로만 분포를 산출하라.
'''


In [ ]:

# -*- coding: utf-8 -*-
import re, json, math, time, random
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import pandas as pd

# ---------------------------
# Utils
# ---------------------------
MONTHS = [f"{y}-{m:02d}" for (y, m) in [(2024,7),(2024,8),(2024,9),(2024,10),(2024,11),(2024,12),
                                        (2025,1),(2025,2),(2025,3),(2025,4),(2025,5),(2025,6)]]

def month_cols():
    return [f"months_since_launch_{i}" for i in range(1, 13)]

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)

def safe_json_load(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

# ---------------------------
# Group & Attribute Parsing
# ---------------------------
SIZE_PAT = re.compile(r'(\d{2,4})\s?g', re.I)

def parse_size(name: str) -> Optional[str]:
    m = SIZE_PAT.search(name or "")
    return (m.group(1) + "g") if m else None

FLAVOR_TOKENS = ["진", "순", "프리미엄", "고소", "매콤"]
def parse_flavor(name: str) -> Optional[str]:
    nm = name or ""
    for tok in FLAVOR_TOKENS:
        if tok in nm:
            return tok
    return None

def detect_group_key(name: str) -> Optional[str]:
    if "참치액" in name:
        if "진" in name:
            return "참치액_진"
        if "순" in name:
            return "참치액_순"
        if "프리미엄" in name:
            return "참치액_프리미엄"
    if "동원맛참" in name:
        return "동원맛참"
    return None

def build_group_index(product_names: List[str]) -> Dict[str, Dict[str, Any]]:
    groups: Dict[str, Dict[str, Any]] = {}
    for p in product_names:
        gk = detect_group_key(p)
        if not gk:
            continue
        size = parse_size(p)
        flv = parse_flavor(p)
        if gk not in groups:
            groups[gk] = {"products": [], "sizes": set(), "flavors": set()}
        groups[gk]["products"].append(p)
        if size: groups[gk]["sizes"].add(size)
        if flv: groups[gk]["flavors"].add(flv)
    for gk in groups:
        groups[gk]["sizes"] = sorted(groups[gk]["sizes"])
        groups[gk]["flavors"] = sorted(groups[gk]["flavors"])
    return groups

# ---------------------------
# Config
# ---------------------------
@dataclass
class Config:
    personas_json: str
    products_csv: str
    sample_submission_csv: str
    output_csv: str
    cache_dir: str
    use_llm: bool = True
    simulate_if_no_key: bool = True
    openai_model: str = "gpt-4o-mini"
    requests_per_minute: int = 120
    temperature: float = 0.2
    top_p: float = 0.95
    persona_limit: Optional[int] = None
    product_limit: Optional[int] = None
    macro_path: List[float] = field(default_factory=lambda: list(np.linspace(1.00, 1.05, num=12)))
    event_rules: List[Dict[str, Any]] = field(default_factory=list)
    line_penalties: Dict[str, float] = field(default_factory=dict)
    ratio_rules: Dict[str, Any] = field(default_factory=dict)  # legacy hard rules (unused by default)
    soft_split: Dict[str, Any] = field(default_factory=lambda: {"alpha_persona": 0.35, "alpha_prior": 0.12, "priors": {}})
    seed: int = 42

# ---------------------------
# Prompt Builder
# ---------------------------
class PromptBuilder:
    def __init__(self, prompt_template: str):
        self.template = prompt_template

    @staticmethod
    def pick_weighted_attrs(persona: Dict[str, Any], min_n: int = 10) -> List[Tuple[str, Any, float]]:
        priority = [
            ("age", 1.2), ("gender", 1.0), ("income", 1.1), ("family_size", 1.1), ("region", 1.0),
            ("occupation", 0.9), ("purchase_channel", 1.0), ("cooking_freq", 1.1), ("brand_loyalty", 1.2),
            ("discount_sensitivity", 1.2), ("flavor_preference", 1.0), ("health_focus", 0.9), ("lifestyle", 1.0),
        ]
        found = []
        for key, base_w in priority:
            if key in persona and persona[key] is not None:
                val = persona[key]
                richness = 1.0 + (0.1 if isinstance(val, (list, dict)) else 0.0)
                found.append((key, val, round(base_w * richness, 3)))
        if len(found) < min_n:
            for k, v in persona.items():
                if any(k == p[0] for p in found): 
                    continue
                if k in {"id", "persona_id", "name", "summary"}:
                    continue
                if v is None:
                    continue
                found.append((k, v, 0.8))
                if len(found) >= min_n:
                    break
        return found[:max(min_n, len(found))]

    def build(self, persona: Dict[str, Any], product: Dict[str, Any], months: List[str], group_info: Optional[Dict[str, Any]] = None) -> str:
        attrs = self.pick_weighted_attrs(persona, min_n=10)
        lines = [f"- {k}: {v} (weight={w})" for k, v, w in attrs]
        persona_block = "\n".join(lines)
        if group_info:
            product = dict(product)
            product["group_variants"] = {
                "sizes": group_info.get("sizes", []),
                "flavors": group_info.get("flavors", []),
            }
        product_block = json.dumps(product, ensure_ascii=False)
        months_block = ", ".join(months)
        prompt = self.template.format(
            persona_block=persona_block,
            product_block=product_block,
            months_block=months_block
        )
        return prompt

# ---------------------------
# LLM Client (OpenAI or simulate)
# ---------------------------
class LLMClient:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.cache_dir = Path(cfg.cache_dir); ensure_dir(self.cache_dir)
        self._openai = None
        if self.cfg.use_llm and ("OPENAI_API_KEY" in os.environ):
            try:
                import openai
                self._openai = openai
                self._openai.api_key = os.environ["OPENAI_API_KEY"]
            except Exception as e:
                print(f"[WARN] OpenAI SDK not available: {e}. Falling back to simulation.")

    def _cache_path(self, persona_id: str, product_name: str) -> Path:
        safe_prod = re.sub(r'[^A-Za-z0-9가-힣_]+', '_', product_name)
        return self.cache_dir / f"{persona_id}__{safe_prod}.json"

    def ask(self, persona_id: str, product_name: str, prompt: str) -> Dict[str, Any]:
        cp = self._cache_path(persona_id, product_name)
        if cp.exists():
            with cp.open("r", encoding="utf-8") as f:
                return json.load(f)

        if (self._openai is None) or (not self.cfg.use_llm):
            data = self.simulate_response(product_name)
            with cp.open("w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            return data

        for attempt in range(5):
            try:
                resp = self._openai.ChatCompletion.create(
                    model=self.cfg.openai_model,
                    temperature=self.cfg.temperature,
                    top_p=self.cfg.top_p,
                    messages=[
                        {"role": "system", "content": "You are a market research assistant that outputs STRICT JSON only."},
                        {"role": "user", "content": prompt},
                    ]
                )
                content = resp["choices"][0]["message"]["content"]
                data = self._extract_json(content)
                with cp.open("w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)
                return data
            except Exception as e:
                wait = 1.5 * (attempt + 1)
                print(f"[WARN] LLM call failed (attempt {attempt+1}): {e}. sleep {wait}s")
                time.sleep(wait)
        data = self.simulate_response(product_name)
        with cp.open("w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        return data

    @staticmethod
    def _extract_json(text: str) -> Dict[str, Any]:
        m = re.search(r'\{.*\}', text, re.S)
        if not m:
            raise ValueError("No JSON object found in LLM output")
        return json.loads(m.group(0))

    @staticmethod
    def base_month_curve() -> np.ndarray:
        x = np.linspace(-2, 2, 12)
        curve = 1 / (1 + np.exp(-x))
        return curve / curve.mean()

    def simulate_response(self, product_name: str) -> Dict[str, Any]:
        base = self.base_month_curve()
        if "참치액" in product_name:
            base = base * np.array([0.9,0.95,1.15,1.05,0.95,0.9,0.92,1.10,0.95,0.9,0.88,0.85])
        if "맛참" in product_name:
            base = base * np.array([0.95,1.0,1.08,1.02,0.98,0.95,1.0,1.05,1.0,0.97,0.95,0.92])
        base = base / base.mean()
        prob = float(np.clip(np.random.normal(0.12, 0.03), 0.02, 0.35))
        qty  = float(np.clip(np.random.normal(1.05, 0.2), 0.5, 2.5))
        months = (base * prob * qty).tolist()
        # synthetic preferences
        size_pref = {"500g": 0.8, "900g": 0.2} if "참치액" in product_name else {"90g": 0.6, "135g": 0.4}
        flv_pref  = {"진": 0.6, "순": 0.35, "프리미엄": 0.05} if "참치액" in product_name else {"고소": 0.65, "매콤": 0.35}
        return {
            "buy_probability": prob,
            "expected_qty_per_purchase": qty,
            "monthly_pattern": [round(float(x), 6) for x in months],
            "pack_size_preference": size_pref,
            "flavor_preference": flv_pref,
        }

# ---------------------------
# Adjustments
# ---------------------------
def apply_line_penalties(product_name: str, value: float, penalties: Dict[str, float]) -> float:
    for key, mult in penalties.items():
        if key in product_name:
            return value * mult
    return value

def apply_event_rules(product_name: str, months: List[str], base: np.ndarray, event_rules: List[Dict[str, Any]]) -> np.ndarray:
    adj = base.copy()
    for rule in event_rules:
        kw = rule.get("keyword", "")
        m = rule.get("month", "")
        factor = float(rule.get("factor", 1.0))
        if kw and kw in product_name and m in months:
            idx = months.index(m); adj[idx] *= factor
    return adj

# ---------------------------
# Runner
# ---------------------------
def run_pipeline(cfg_dict: dict, prompt_template: str):
    cfg = Config(**cfg_dict)
    seed_everything(cfg.seed)

    # Load personas / products / sample
    pj = Path(cfg.personas_json)
    if pj.exists():
        personas = safe_json_load(pj)
    else:
        # DEMO fallback: synthesize small personas (so notebook runs immediately)
        personas = [{"id": f"demo_{i}", "age": 40+i%10, "gender": "여", "family_size": 3+(i%2),
                     "income": "middle", "purchase_channel": "대형마트", "brand_loyalty": 0.6, 
                     "discount_sensitivity": 0.4, "cooking_freq": "주 2~3회",
                     "flavor_preference": ["고소","순"], "lifestyle": "건강 중시"} for i in range(50)]
        print("[INFO] personas_json not found → DEMO personas(50) generated.")

    prod_path = Path(cfg.products_csv)
    sub_path  = Path(cfg.sample_submission_csv)
    if prod_path.exists() and sub_path.exists():
        products = pd.read_csv(prod_path)
        sample   = pd.read_csv(sub_path)
        prod_names = list(sample["product_name"])
    else:
        # DEMO fallback: tiny products & submission
        prod_names = ["동원참치액 진 500g", "동원참치액 진 900g", "동원참치액 순 500g", "동원참치액 순 900g",
                      "동원맛참 고소 90g", "동원맛참 매콤 90g"]
        products = pd.DataFrame({"product_name": prod_names, "product_feature": ["demo"]*len(prod_names)})
        sample = pd.DataFrame({"product_name": prod_names, **{f"months_since_launch_{i}":[0]*len(prod_names) for i in range(1,13)}})
        print("[INFO] products_csv / sample_submission_csv not found → DEMO products/submission created.")

    if cfg.persona_limit:
        personas = personas[:cfg.persona_limit]
    if cfg.product_limit:
        prod_names = prod_names[:cfg.product_limit]

    prod_map = {str(r["product_name"]): r._asdict() if hasattr(r, "_asdict") else dict(r) for _, r in products.iterrows()}

    pb  = PromptBuilder(prompt_template)
    llm = LLMClient(cfg)

    mcols = month_cols()
    mat = {p: np.zeros(12, dtype=float) for p in prod_names}

    # Build group index & preference accumulators
    groups = build_group_index(prod_names)
    pref_counts = {gk: {"size": {}, "flavor": {}, "weight_sum": 0.0} for gk in groups.keys()}

    rpm = max(1, cfg.requests_per_minute); interval = 60.0 / float(rpm); last_ts = 0.0

    for p in prod_names:
        pinfo = prod_map.get(p, {"product_name": p})
        gk = detect_group_key(p); ginfo = groups.get(gk, None)
        for idx, persona in enumerate(personas):
            pid = str(persona.get("id") or persona.get("persona_id") or f"p{idx}")
            prompt = pb.build(persona, pinfo, MONTHS, group_info=ginfo)
            data = llm.ask(pid, p, prompt)
            vec = np.array(data.get("monthly_pattern", [0]*12), dtype=float)
            mat[p] += vec
            w = float(data.get("buy_probability", 0.0)) * float(data.get("expected_qty_per_purchase", 0.0))
            if gk and gk in pref_counts:
                if isinstance(data.get("pack_size_preference"), dict):
                    for k, v in data["pack_size_preference"].items():
                        pref_counts[gk]["size"][k] = pref_counts[gk]["size"].get(k, 0.0) + float(v) * (w + 1e-6)
                if isinstance(data.get("flavor_preference"), dict):
                    for k, v in data["flavor_preference"].items():
                        pref_counts[gk]["flavor"][k] = pref_counts[gk]["flavor"].get(k, 0.0) + float(v) * (w + 1e-6)
                pref_counts[gk]["weight_sum"] += (w + 1e-6)
            now = time.time()
            if now - last_ts < interval:
                time.sleep(max(0.0, interval - (now - last_ts)))
            last_ts = time.time()

    out = pd.DataFrame({"product_name": prod_names})
    for i in range(12):
        out[f"months_since_launch_{i+1}"] = 0.0
    for p in prod_names:
        out.loc[out["product_name"] == p, mcols] = mat[p]

    macro = np.array(cfg.macro_path, dtype=float)
    out[mcols] = (out[mcols].astype(float) * macro).values

    # events + line penalties
    for i, row in out.iterrows():
        pname = row["product_name"]
        vec = row[mcols].astype(float).values
        vec = apply_event_rules(pname, MONTHS, vec, cfg.event_rules)
        vec2 = np.array([apply_line_penalties(pname, float(v), cfg.line_penalties) for v in vec])
        out.loc[i, mcols] = vec2

    # Soft split per group
    def _normalize(d: Dict[str, float]) -> Dict[str, float]:
        tot = sum(max(0.0, v) for v in d.values())
        if tot <= 0: 
            return {k: 1.0/len(d) for k in d} if d else {}
        return {k: max(0.0, v)/tot for k, v in d.items()}

    group_weights = {}
    for gk, acc in pref_counts.items():
        sz = _normalize(acc["size"]) if acc["size"] else {}
        fl = _normalize(acc["flavor"]) if acc["flavor"] else {}
        group_weights[gk] = {"size": sz, "flavor": fl}

    ss = cfg.soft_split or {}
    alpha_persona = float(ss.get("alpha_persona", 0.35))
    alpha_prior   = float(ss.get("alpha_prior", 0.12))
    priors        = ss.get("priors", {})

    def sku_weight(pname: str, gk: str) -> float:
        size = parse_size(pname) or ""
        flv  = parse_flavor(pname) or ""
        gw = group_weights.get(gk, {})
        w_size = gw.get("size", {}).get(size, 0.0)
        w_flv  = gw.get("flavor", {}).get(flv, 0.0)
        pr = priors.get(gk, {})
        ps = pr.get("size", {}).get(size, None)
        pf = pr.get("flavor", {}).get(flv, None)
        if ps is not None:
            w_size = (1 - alpha_prior) * w_size + alpha_prior * float(ps)
        if pf is not None:
            w_flv  = (1 - alpha_prior) * w_flv  + alpha_prior * float(pf)
        val = (w_size if w_size > 0 else 1.0) * (w_flv if w_flv > 0 else 1.0)
        return float(val)

    for gk, ginfo in groups.items():
        idxs = out[out["product_name"].isin(ginfo["products"])].index
        if len(idxs) < 2:
            continue
        cur = out.loc[idxs, mcols].astype(float).values
        base_tot = cur.sum(axis=0)
        raw_w = np.array([sku_weight(out.at[i, "product_name"], gk) for i in idxs], dtype=float)
        if raw_w.sum() <= 0:
            continue
        w = raw_w / raw_w.sum()
        base_share = np.divide(cur, np.maximum(base_tot, 1e-9), where=np.broadcast_to(base_tot, cur.T.shape).T)
        new_share = (1 - alpha_persona) * base_share + alpha_persona * w.reshape(-1, 1)
        new_vals  = (new_share * base_tot.reshape(1, -1))
        out.loc[idxs, mcols] = np.maximum(0, np.round(new_vals))

    # Finalize
    out[mcols] = out[mcols].clip(lower=0).round(0)
    for c in mcols:
        out[c] = out[c].astype(int)

    out = out[["product_name"] + mcols]
    Path(cfg.output_csv).parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(cfg.output_csv, index=False, encoding="utf-8-sig")
    return out


In [ ]:

# 실행
from pathlib import Path
import pandas as pd

df_out = run_pipeline(CFG, PROMPT_TEMPLATE)
print("Saved:", CFG["output_csv"])

# 미리보기
import caas_jupyter_tools
caas_jupyter_tools.display_dataframe_to_user("Submission Preview (head)", df_out.head(10))
df_out.head(10)  # notebook 안에서도 표시
